<a href="https://colab.research.google.com/github/Vojta1999/Programovani1/blob/master/exercise_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6: Named Entity Recognition & Entity Linking

## Scenario

You are a data analyst tasked with building an **information extraction pipeline**. You receive a pile of unlabeled texts from a specific domain, and your manager asks:

> *"What entities are mentioned in these texts? Can you link them to a knowledge base so we can assign them to items from a structured database?"*

In this lab you will:

1. **Load domain texts** from a prepared dataset
2. **Run a generic NER model** and observe its limitations on domain text
3. **Run a domain-specific NER model** and compare the improvement
4. **Link extracted entities to Wikidata** to build an enriched knowledge table

You will choose **one domain** to work with:

| Domain | Domain NER Model | Entity Types |
|--------|------------------|---------------|
| **Sports News** | `Livesport/xx_ner_sport_entities_uncased` (spaCy) | PLAYER, TEAM, TOURNAMENT |
| **Recipes** | `Dizex/InstaFoodRoBERTa-NER` (HuggingFace) | FOOD |

These domain-specific NER models are fine-tuned models built from pre-trained general models using annotated datasets from respective domains.

## 1: Setup & Data Loading

### Task 1: Choose Your Domain

Set the `DOMAIN` variable below to `"sports"` or `"recipes"`. This will control which dataset and models are used throughout the notebook.

In [ ]:
# TASK 1: Choose your domain
# Options: "sports" or "recipes"

DOMAIN = "sports"

assert DOMAIN in ("sports", "recipes"), "DOMAIN must be 'sports' or 'recipes'"
print(f"Selected domain: {DOMAIN}")

In [ ]:
# Install dependencies
!pip install -q transformers torch requests pandas

# Install domain-specific dependencies
if DOMAIN == "sports":
    !pip install -q spacy spacy-transformers
    # The sports model will be downloaded automatically when first loaded
elif DOMAIN == "recipes":
    pass  # InstaFoodRoBERTa downloads automatically via HuggingFace

In [ ]:
import pandas as pd
from transformers import pipeline
import requests
import time
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80)

In [ ]:
SPORTS_TEXTS = [
    "Lionel Messi scored twice as Barcelona crushed Real Madrid 5-0 in El Clasico at Camp Nou. Xavi and Andres Iniesta controlled the midfield throughout.",
    "Cristiano Ronaldo completed a hat trick in the Champions League quarter-final as Manchester United defeated Porto 3-1 at Old Trafford. Sir Alex Ferguson praised his star forward after the match.",
    "Bayern Munich signed Robert Lewandowski from Borussia Dortmund on a free transfer. The Polish striker scored 20 goals in the Bundesliga last season.",
    "Chelsea defeated Liverpool 2-1 in the Premier League at Stamford Bridge. Didier Drogba scored the winner in stoppage time, leaving Rafael Benitez frustrated on the sideline.",
    "The World Cup final between France and Croatia ended 4-2 in Moscow. Kylian Mbappe became only the second teenager after Pele to score in a World Cup final.",
    "AC Milan and Inter Milan battled to a 1-1 draw in the Derby della Madonnina at San Siro. Zlatan Ibrahimovic equalized for Milan in the second half after Lautaro Martinez opened the scoring.",
    "Neymar transferred from Barcelona to Paris Saint-Germain for a world-record fee of 222 million euros. The Brazilian forward signed a five-year contract with the Ligue 1 club.",
    "Erling Haaland scored a brace on his Manchester City debut against West Ham United in the Premier League. Pep Guardiola called the Norwegian striker a once-in-a-generation talent.",
    "Juventus won the Serie A title for the ninth consecutive season. Gianluigi Buffon lifted the trophy in his final match before retiring from the club.",
    "The Europa League final saw Sevilla defeat Inter Milan 3-2 in Cologne. Diego Carlos scored an overhead kick to seal the victory for the Spanish side.",
    "Sidney Crosby scored the overtime winner as the Pittsburgh Penguins defeated the Detroit Red Wings 4-3 in Game 7 of the Stanley Cup Finals.",
    "The Edmonton Oilers traded Wayne Gretzky to the Los Angeles Kings in a blockbuster deal that shocked the entire NHL. Gretzky had led the Oilers to four Stanley Cup championships.",
    "Alexander Ovechkin netted his 800th career goal as the Washington Capitals beat the Chicago Blackhawks 4-1 at Capital One Arena. He is now third on the all-time NHL scoring list.",
    "The Montreal Canadiens upset the heavily favored Toronto Maple Leafs in seven games during the first round of the NHL Playoffs. Carey Price was outstanding in goal throughout the series.",
    "Connor McDavid recorded four assists as the Edmonton Oilers crushed the Calgary Flames 6-2 in the Battle of Alberta. Leon Draisaitl added two goals in the dominant performance.",
    "The Tampa Bay Lightning won back-to-back Stanley Cup titles by defeating the Montreal Canadiens four games to one. Andrei Vasilevskiy was named the playoff MVP after posting a 1.90 goals-against average.",
    "Jaromir Jagr announced his retirement from the NHL after a legendary 24-season career. The Czech forward scored 766 goals and recorded 1,155 assists across stints with the Penguins, Capitals, Rangers, and Panthers.",
    "The Boston Bruins acquired defenseman Ray Bourque from the Colorado Avalanche in a late-season trade. Bourque finally won his first Stanley Cup after 22 years in the league.",
    "Auston Matthews scored his 60th goal of the season as the Toronto Maple Leafs clinched first place in the Atlantic Division. Matthews became the first Maple Leaf to reach 60 goals since the 1993-94 season.",
    "The New York Rangers defeated the Vancouver Canucks 3-2 in Game 7 of the Stanley Cup Finals at Madison Square Garden. Mark Messier guaranteed the victory before the game and delivered on his promise.",
]

RECIPE_TEXTS = [
    "No-Bake Nut Cookies\n\nIngredients:\n- 1 c. firmly packed brown sugar\n- 1/2 c. evaporated milk\n- 1/2 tsp. vanilla\n- 1/2 c. broken nuts (pecans)\n- 2 Tbsp. butter or margarine\n- 3 1/2 c. bite size shredded rice biscuits\n\nDirections:\nIn a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter. Stir over medium heat until mixture boils. Cook 5 minutes, stirring frequently. Remove from heat and stir in vanilla and cereal. Drop by teaspoonfuls onto wax paper. Let stand until firm, about 30 minutes.",
    "Classic Chicken Parmesan\n\nIngredients:\n- 4 boneless chicken breasts\n- 1 cup breadcrumbs\n- 1/2 cup parmesan cheese\n- 2 cups marinara sauce\n- 1 cup mozzarella cheese\n- 2 eggs\n- salt and pepper\n\nDirections:\nPound chicken breasts to even thickness. Dip in beaten eggs, then coat with breadcrumbs mixed with parmesan. Pan-fry in olive oil until golden. Top with marinara sauce and mozzarella cheese. Bake at 400F for 15 minutes until cheese is melted.",
    "Creamy Tomato Basil Soup\n\nIngredients:\n- 2 cans diced tomatoes\n- 1 cup heavy cream\n- 1/4 cup fresh basil\n- 3 cloves garlic\n- 1 onion\n- 2 cups chicken broth\n- 2 Tbsp. olive oil\n- salt and pepper to taste\n\nDirections:\nSaute onion and garlic in olive oil until soft. Add tomatoes and chicken broth. Simmer for 20 minutes. Blend until smooth. Stir in heavy cream and chopped basil. Season with salt and pepper.",
    "Homemade Guacamole\n\nIngredients:\n- 3 ripe avocados\n- 1 lime, juiced\n- 1 tsp. salt\n- 1/2 cup onion, diced\n- 3 Tbsp. fresh cilantro\n- 2 roma tomatoes\n- 1 tsp. garlic\n- 1 pinch cayenne pepper\n\nDirections:\nCut avocados in half and remove pit. Scoop out flesh and mash with a fork. Stir in lime juice and salt. Mix in onion, cilantro, tomatoes, and garlic. Add cayenne pepper. Refrigerate for 1 hour before serving with tortilla chips.",
    "Banana Bread\n\nIngredients:\n- 3 ripe bananas\n- 1/3 cup melted butter\n- 3/4 cup sugar\n- 1 egg\n- 1 tsp. vanilla extract\n- 1 tsp. baking soda\n- pinch of salt\n- 1 1/2 cups all-purpose flour\n\nDirections:\nPreheat oven to 350F. Mash the bananas with a fork. Mix in melted butter, sugar, egg, and vanilla. Add baking soda, salt, and flour. Pour into a greased loaf pan. Bake for 60-65 minutes until golden brown.",
    "Grilled Salmon with Lemon Dill Sauce\n\nIngredients:\n- 4 salmon fillets\n- 2 Tbsp. olive oil\n- 1 lemon\n- 2 Tbsp. fresh dill\n- 1/2 cup sour cream\n- salt and pepper\n\nDirections:\nBrush salmon with olive oil, season with salt and pepper. Grill over medium-high heat for 4-5 minutes per side. For the sauce, mix sour cream, lemon juice, and chopped dill. Serve salmon topped with lemon dill sauce.",
    "Beef Tacos\n\nIngredients:\n- 1 lb ground beef\n- 1 packet taco seasoning\n- 8 taco shells\n- 1 cup shredded lettuce\n- 1 cup shredded cheddar cheese\n- 1/2 cup sour cream\n- 1 cup salsa\n- 1 tomato, diced\n\nDirections:\nBrown ground beef in a skillet over medium heat. Drain fat and add taco seasoning with water. Simmer for 5 minutes. Warm taco shells in oven. Fill shells with beef, lettuce, cheese, tomato, sour cream, and salsa.",
    "Mushroom Risotto\n\nIngredients:\n- 1 1/2 cups arborio rice\n- 4 cups chicken broth\n- 1 cup white wine\n- 8 oz mushrooms, sliced\n- 1/2 cup parmesan cheese\n- 1 onion, diced\n- 3 Tbsp. butter\n- 2 Tbsp. olive oil\n\nDirections:\nSaute onion in olive oil and butter until translucent. Add mushrooms and cook until tender. Add arborio rice and toast for 2 minutes. Pour in white wine and stir until absorbed. Add broth one ladle at a time, stirring constantly. Finish with parmesan cheese.",
    "Chocolate Chip Cookies\n\nIngredients:\n- 2 1/4 cups flour\n- 1 tsp. baking soda\n- 1 cup butter, softened\n- 3/4 cup sugar\n- 3/4 cup brown sugar\n- 2 eggs\n- 1 tsp. vanilla\n- 2 cups chocolate chips\n\nDirections:\nPreheat oven to 375F. Mix flour and baking soda. In a separate bowl, cream butter, sugar, and brown sugar. Beat in eggs and vanilla. Gradually blend in flour mixture. Stir in chocolate chips. Drop by spoonfuls onto baking sheets. Bake 9-11 minutes until golden.",
    "Caesar Salad\n\nIngredients:\n- 1 head romaine lettuce\n- 1/2 cup croutons\n- 1/4 cup parmesan cheese\n- 2 Tbsp. lemon juice\n- 1 clove garlic\n- 1 tsp. Worcestershire sauce\n- 1/2 cup olive oil\n- 1 egg yolk\n- salt and pepper\n\nDirections:\nWash and chop romaine lettuce. For dressing, whisk together egg yolk, lemon juice, garlic, and Worcestershire sauce. Slowly drizzle in olive oil while whisking. Toss lettuce with dressing, croutons, and parmesan cheese.",
    "Spaghetti Bolognese\n\nIngredients:\n- 1 lb spaghetti\n- 1 lb ground beef\n- 1 can crushed tomatoes\n- 1 onion\n- 3 cloves garlic\n- 1/2 cup red wine\n- 2 Tbsp. olive oil\n- 1 carrot\n- 1 celery stalk\n- fresh basil\n\nDirections:\nCook spaghetti according to package directions. Saute onion, garlic, carrot, and celery in olive oil. Add ground beef and brown. Pour in crushed tomatoes and red wine. Simmer for 30 minutes. Serve sauce over spaghetti with fresh basil.",
    "Blueberry Muffins\n\nIngredients:\n- 1 1/2 cups flour\n- 3/4 cup sugar\n- 1/3 cup vegetable oil\n- 1 egg\n- 1/3 cup milk\n- 1 cup fresh blueberries\n- 2 tsp. baking powder\n- 1/2 tsp. salt\n\nDirections:\nPreheat oven to 400F. Mix flour, sugar, baking powder, and salt. In another bowl, combine oil, egg, and milk. Stir wet ingredients into dry ingredients until just combined. Fold in blueberries. Fill muffin cups 2/3 full. Bake 20-25 minutes.",
    "Garlic Shrimp Pasta\n\nIngredients:\n- 1 lb shrimp, peeled\n- 12 oz linguine\n- 4 cloves garlic, minced\n- 1/2 cup white wine\n- 2 Tbsp. butter\n- 2 Tbsp. olive oil\n- red pepper flakes\n- fresh parsley\n- lemon juice\n\nDirections:\nCook linguine al dente. Saute garlic in olive oil and butter for 1 minute. Add shrimp and cook until pink. Pour in white wine and lemon juice. Toss with cooked linguine. Garnish with parsley and red pepper flakes.",
    "Vegetable Stir Fry\n\nIngredients:\n- 2 cups broccoli florets\n- 1 red bell pepper\n- 1 cup snap peas\n- 2 carrots, sliced\n- 3 Tbsp. soy sauce\n- 1 Tbsp. sesame oil\n- 2 cloves garlic\n- 1 Tbsp. ginger\n- 1 Tbsp. cornstarch\n\nDirections:\nHeat sesame oil in a wok over high heat. Add garlic and ginger, stir for 30 seconds. Add carrots and broccoli, cook 3 minutes. Add bell pepper and snap peas. Mix soy sauce with cornstarch and pour over vegetables. Stir until sauce thickens.",
    "Pancakes\n\nIngredients:\n- 1 1/2 cups flour\n- 3 1/2 tsp. baking powder\n- 1 Tbsp. sugar\n- 1 1/4 cups milk\n- 1 egg\n- 3 Tbsp. melted butter\n- maple syrup for serving\n\nDirections:\nMix flour, baking powder, sugar, and salt. Make a well and pour in milk, egg, and melted butter. Mix until smooth. Heat a griddle over medium-high heat. Pour batter onto griddle. Cook until bubbles form, then flip. Serve with maple syrup and fresh berries.",
    "Greek Salad\n\nIngredients:\n- 2 cucumbers\n- 4 tomatoes\n- 1 red onion\n- 1 cup kalamata olives\n- 1 cup feta cheese\n- 2 Tbsp. olive oil\n- 1 Tbsp. red wine vinegar\n- 1 tsp. oregano\n\nDirections:\nCut cucumbers, tomatoes, and red onion into chunks. Add kalamata olives. Crumble feta cheese on top. Drizzle with olive oil and red wine vinegar. Sprinkle with oregano, salt, and pepper. Toss gently and serve.",
    "BBQ Pulled Pork\n\nIngredients:\n- 3 lb pork shoulder\n- 1 cup BBQ sauce\n- 1/2 cup apple cider vinegar\n- 1 Tbsp. brown sugar\n- 1 tsp. paprika\n- 1 tsp. garlic powder\n- hamburger buns\n- coleslaw\n\nDirections:\nRub pork shoulder with paprika, garlic powder, salt, and pepper. Place in slow cooker with apple cider vinegar and brown sugar. Cook on low for 8 hours. Shred pork with two forks. Mix with BBQ sauce. Serve on hamburger buns with coleslaw.",
    "French Onion Soup\n\nIngredients:\n- 4 large onions\n- 4 cups beef broth\n- 1 cup white wine\n- 3 Tbsp. butter\n- 1 Tbsp. flour\n- 4 slices French bread\n- 1 cup gruyere cheese\n- fresh thyme\n\nDirections:\nSlice onions thinly. Cook in butter over low heat for 45 minutes until caramelized. Sprinkle with flour, stir 1 minute. Add white wine and beef broth. Simmer 20 minutes with thyme. Ladle into oven-safe bowls, top with French bread and gruyere cheese. Broil until bubbly.",
    "Lemon Garlic Roasted Chicken\n\nIngredients:\n- 1 whole chicken (4 lbs)\n- 2 lemons\n- 6 cloves garlic\n- 3 Tbsp. olive oil\n- 2 Tbsp. butter\n- fresh rosemary\n- fresh thyme\n- salt and pepper\n\nDirections:\nPreheat oven to 425F. Pat chicken dry. Stuff cavity with lemon halves, garlic, and herbs. Rub outside with olive oil and butter. Season generously with salt and pepper. Roast for 1 hour 15 minutes until internal temperature reaches 165F. Let rest 10 minutes before carving.",
    "Mango Smoothie\n\nIngredients:\n- 1 cup frozen mango chunks\n- 1 banana\n- 1 cup yogurt\n- 1/2 cup orange juice\n- 1 Tbsp. honey\n\nDirections:\nCombine frozen mango, banana, yogurt, orange juice, and honey in a blender. Blend until smooth. Pour into glasses and serve immediately. For a thicker smoothie, add more frozen mango or a handful of ice cubes.",
]

# Select texts based on chosen domain
texts = SPORTS_TEXTS if DOMAIN == "sports" else RECIPE_TEXTS

print(f"Loaded {len(texts)} texts from the '{DOMAIN}' domain.\n")

for i in range(3):
    print(f"--- Text {i+1} ---")
    print(texts[i][:300])
    print()

## 2: Generic NER — The Baseline

Let's start with a **generic NER model**: [`dslim/bert-base-NER`](https://huggingface.co/dslim/bert-base-NER).

This model was trained on the **CoNLL-2003** dataset (news articles) and recognizes 4 standard entity types:

| Label | Meaning |
|-------|----------|
| **PER** | Person names |
| **ORG** | Organizations |
| **LOC** | Locations |
| **MISC** | Miscellaneous (events, nationalities, etc.) |

In [ ]:
# Load the generic NER model
generic_ner = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"  # merges subword tokens
)

print("Generic NER model loaded.")

In [ ]:
# Run generic NER on the first 2 texts

for i in range(2):
    text = texts[i]
    print(f"=== Text {i+1} ===")
    print(text[:300])
    print()

    results = generic_ner(text)
    if results:
        print(f"{'Entity':<35} {'Type':<8} {'Score'}")
        print("-" * 55)
        for ent in results:
            print(f"{ent['word']:<35} {ent['entity_group']:<8} {ent['score']:.3f}")
    else:
        print("  (no entities found)")
    print()

### Task 2: Analyze Generic NER Results

Look at the entities found above and answer the following questions:

1. What types of entities (PER, ORG, LOC, MISC) did the generic model find?
2. What **domain-specific information** is clearly present in the text but was **NOT detected**?
   - For *sports*: Were any player names missed? Team nicknames? Tournament names?
   - For *recipes*: Were any ingredients detected? Cooking techniques? Equipment?

**Your answers:**

1. ...

2. ...


## Part 3: Domain-Specific NER — The Upgrade

Now let's use a model that was **fine-tuned** specifically for your domain. This demonstrates the power of fine-tuning — same underlying transformer architecture, but trained on domain-specific annotated data.

| Domain | Model | Entity Types | Framework |
|--------|-------|--------------|------------|
| Sports | [`Livesport/xx_ner_sport_entities_uncased`](https://huggingface.co/Livesport/xx_ner_sport_entities_uncased) | PLAYER, TEAM, TOURNAMENT, ALIAS_TEAM | spaCy |
| Recipes | [`Dizex/InstaFoodRoBERTa-NER`](https://huggingface.co/Dizex/InstaFoodRoBERTa-NER) | FOOD | HuggingFace |

Notice that the sports model uses **spaCy** while the recipes model uses **HuggingFace Transformers**. In the real world, the best model for your domain may live in a different framework.

In [ ]:
# Load the domain-specific NER model

if DOMAIN == "sports":
    import spacy
    from huggingface_hub import snapshot_download

    # Download and load the spaCy sports NER model from HuggingFace Hub
    model_path = snapshot_download("Livesport/xx_ner_sport_entities_uncased")
    nlp_domain = spacy.load(model_path)

    def run_domain_ner(text):
        """Run sports NER using spaCy model."""
        doc = nlp_domain(text)
        return [{"word": ent.text, "entity_group": ent.label_, "score": 1.0} for ent in doc.ents]

    print("Sports NER model loaded (spaCy).")
    print("Entity types: PLAYER, TEAM, TOURNAMENT, ALIAS_TEAM")

elif DOMAIN == "recipes":
    _food_ner = pipeline(
        "ner",
        model="Dizex/InstaFoodRoBERTa-NER",
        aggregation_strategy="simple"
    )

    def _merge_entities(entities, original_text):
        """Merge adjacent entities of the same type (fixes subword splitting)."""
        if not entities:
            return entities
        merged = [entities[0].copy()]
        for ent in entities[1:]:
            prev = merged[-1]
            if ent["entity_group"] == prev["entity_group"] and ent["start"] <= prev["end"] + 1:
                prev["end"] = ent["end"]
                prev["word"] = original_text[prev["start"]:prev["end"]].strip()
                prev["score"] = min(prev["score"], ent["score"])
            else:
                merged.append(ent.copy())
        for e in merged:
            e["word"] = e["word"].strip()
        return merged

    def run_domain_ner(text):
        """Run food NER using HuggingFace model with subword merging."""
        raw = _food_ner(text)
        merged = _merge_entities(raw, text)
        return [{"word": r["word"], "entity_group": r["entity_group"], "score": r["score"]} for r in merged]

    print("Food NER model loaded (HuggingFace).")
    print("Entity types: FOOD")

In [ ]:
# Side-by-side comparison: Generic vs Domain-Specific NER

model_name = "spaCy sports model" if DOMAIN == "sports" else "InstaFoodRoBERTa"

for i in range(2):
    text = texts[i]
    print("=" * 60)
    print(f"Text {i+1}: {text[:150]}...")
    print("=" * 60)

    # Generic NER
    generic_results = generic_ner(text)
    print(f"\n  Generic NER (dslim/bert-base-NER):")
    if generic_results:
        for ent in generic_results:
            print(f"    {ent['word']:<35} {ent['entity_group']}")
    else:
        print("    (no entities found)")

    # Domain NER
    domain_results = run_domain_ner(text)
    print(f"\n  Domain NER ({model_name}):")
    if domain_results:
        for ent in domain_results:
            print(f"    {ent['word']:<35} {ent['entity_group']}")
    else:
        print("    (no entities found)")

    print()

### Task 3: Batch Processing & Analysis

Run the domain-specific NER model on **all 20 texts** and analyze the results.

Print:
1. Total number of entities found
2. Number of unique entities found
3. Distribution of entity types
4. Top 15 most frequent entities

In [ ]:
all_entities = []

# Run domain NER on all texts and collect entities

df_entities = pd.DataFrame(all_entities)

## Part 4: Entity Linking to Wikidata

NER gives us entity **strings** like `"Manchester United"` or `"brown sugar"`. But a string alone is ambiguous:
- Does **"Paris"** refer to the capital of France, the city in Texas, or Paris Hilton?
- Does **"Apple"** mean the fruit or the tech company?

**Entity Linking (EL)** solves this by mapping each mention to a unique identifier in a **Knowledge Base** like [Wikidata](https://www.wikidata.org/).

### The EL Pipeline
1. **Candidate Generation** — search the KB for possible matches
2. **Candidate Ranking** — score candidates by relevance/context
3. **Candidate Selection** — pick the best match

We'll use the **Wikidata API** (`wbsearchentities`) which is free and requires no authentication.

In [ ]:
def search_wikidata(query, language="en", limit=5):
    """Search Wikidata for entities matching the query string.

    Returns a list of candidate entities with their IDs and descriptions.
    """
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "search": query,
        "language": language,
        "format": "json",
        "limit": limit
    }
    headers = {"User-Agent": "TextAnalyticsLab/1.0 (university course exercise)"}
    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
        response.raise_for_status()
        return response.json().get("search", [])
    except requests.RequestException as e:
        print(f"  API error for '{query}': {e}")
        return []


# Demo: search for the most frequent entity
demo_entity = df_entities["entity"].value_counts().index[0]
print(f"Searching Wikidata for: '{demo_entity}'\n")

candidates = search_wikidata(demo_entity)
print(f"{'ID':<12} {'Label':<30} Description")
print("-" * 80)
for c in candidates:
    print(f"{c['id']:<12} {c.get('label', 'N/A'):<30} {c.get('description', 'N/A')[:50]}")

### Task 4: Link Entities to Wikidata

Your task: complete the code below to link the **top 15 most frequent entities** to Wikidata. The main goal is to design and implement the most promising candidate selection strategy.

For each entity:
1. Call `search_wikidata(entity_text)` to get candidates
2. Wait briefly between calls: `time.sleep(0.2)` (API rate limiting)
3. If candidates are found, design a strategy to select the most promising one and extract its `id`, `label`, and `description`
4. If no candidates are found, mark as `"NOT_FOUND"`
5. Append a dict with keys: `entity`, `ner_type`, `frequency`, `wikidata_id`, `wikidata_label`, `wikidata_description` to `linked_results`.
6. Produce a pandas dataframe with details from `linked_results`.

In [ ]:
# Prepare unique entities sorted by frequency
unique_entities = (
    df_entities
    .groupby("entity")
    .agg(type=("type", "first"), frequency=("entity", "count"), avg_score=("score", "mean"))
    .reset_index()
    .sort_values("frequency", ascending=False)
)

print(f"Unique entities to link: {len(unique_entities)}")
print(f"Linking top 15...\n")

# TASK 4: Complete the entity linking loop below

linked_results = []

for _, row in unique_entities.head(15).iterrows():
    entity_text = row["entity"]

    # --- YOUR CODE HERE ---
    # Step 1: Search Wikidata for this entity using search_wikidata()
    # Step 2: Add time.sleep(0.2) for rate limiting
    # Step 3: Pick the top candidate or mark as NOT_FOUND
    # Step 4: Append a dict with keys: entity, ner_type, frequency,
    #         wikidata_id, wikidata_label, wikidata_description
    # --- END YOUR CODE ---

    pass

# Build and display the enriched DataFrame
df_linked = pd.DataFrame(linked_results)
df_linked


## Part 5: Discussion

Review your enriched table above and answer the following questions:

1. **Entity Linking Quality**: Were any Wikidata links incorrect (wrong entity matched)? Why might your strategy fail? How would you improve it?

**Your answers:**

1. ...
